# Train the text recognizer (ViT → BERT) on Colab

Runtime → Change runtime type → **GPU** before running.

Trains on word crops cut from the ground-truth boxes of `MyDrive/dataset_receipt/` (`images/` + `metadata.pkl`). Checkpoints go to `MyDrive/document-processing/checkpoints/trocr/{best,last}`.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/dataset_receipt'
TROCR_DIR = '/content/drive/MyDrive/document-processing/checkpoints/trocr'  # on Drive so it survives disconnects

In [ ]:
import os

# For a private repo, add a GITHUB_TOKEN secret in the Colab sidebar (key icon). Never hardcode it here.
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None

auth = f"{token}@" if token else ""
REPO_URL = f"https://{auth}github.com/alexisvannson/document-processing.git"
REPO_DIR = '/content/document-processing'
BRANCH = 'main'

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# Colab already ships torch/torchvision with CUDA; this installs the rest.
!pip install -q -r requirements.txt

In [ ]:
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# Copy the dataset to local disk: reading hundreds of images from Drive every epoch is very slow.
!rsync -a --info=progress2 {DRIVE_DIR}/ /content/dataset_receipt/
!ls /content/dataset_receipt/images | wc -l

## Train

In [ ]:
TROCR_EPOCHS = 30
TROCR_BATCH_SIZE = 16  # ~225M params at 384x384: lower this on CUDA OOM
TROCR_LR = 5e-5

!python train_trocr.py \
    --metadata /content/dataset_receipt/metadata.pkl \
    --img-dir /content/dataset_receipt/images \
    --epochs {TROCR_EPOCHS} --batch-size {TROCR_BATCH_SIZE} --lr {TROCR_LR} \
    --num-workers 2 \
    --out-dir {TROCR_DIR}

In [ ]:
!ls -lh {TROCR_DIR}/best

## Evaluate the best checkpoint

Greedy decoding on the 2,356 validation words. Works without re-running training (setup cells only).

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import VisionEncoderDecoderModel

from dataset import get_recognition_dataloaders
from models.BERT import CharTokenizer
from train_trocr import evaluate, get_autocast

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = CharTokenizer(f'{TROCR_DIR}/best')
model = VisionEncoderDecoderModel.from_pretrained(f'{TROCR_DIR}/best').to(device).eval()

_, valid_loader = get_recognition_dataloaders(
    tokenizer, '/content/dataset_receipt/metadata.pkl', '/content/dataset_receipt/images',
    image_size=model.config.encoder.image_size, batch_size=32, num_workers=2)
autocast, _ = get_autocast(device)
val_loss, cer, word_acc, samples = evaluate(model, valid_loader, tokenizer, device, autocast)
print(f'Val words: {len(samples)} | CER {cer:.4f} | word accuracy {word_acc:.3f}')

In [ ]:
def show(indices, title, cols=4):
    """Word crops with ground truth and prediction."""
    rows = int(np.ceil(len(indices) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 1.8 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, i in zip(axes.flat, indices):
        pred, text = samples[i]
        ax.imshow(valid_loader.dataset.crops[i])
        ax.set_title(f'gt {text!r}\npred {pred!r}', color='green' if pred == text else 'red', fontsize=11)
    fig.suptitle(title, fontsize=15)
    plt.tight_layout()
    plt.show()

errors = [i for i, (pred, text) in enumerate(samples) if pred != text]
print(f'{len(errors)} wrong words out of {len(samples)}')
show(errors[:16], 'Mistakes')
show([i for i in range(len(samples)) if i not in set(errors)][:8], 'Correct')